# Consumer Kafka con Spark Structured Streaming

Cuaderno base para consumir el topic `orden-eventos` desde Spark y parsear el JSON
con un esquema común que funcione tanto si el productor fue hecho en **Java** como en **Python**.

## Objetivo
1. Conectarse a Kafka
2. Leer mensajes del topic `orden-eventos`
3. Convertir `value` a texto
4. Parsear el JSON
5. Filtrar eventos válidos
6. Mostrar en consola los eventos con `total > 100`

## Datos del entorno
- Kafka broker interno: `ec-kafka:9092`
- Topic: `orden-eventos`
- Spark: paquete Kafka para Spark SQL

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaSparkOrdenes2") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-13465377-477d-44c5-a6aa-9748f699e4e9;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

In [2]:
print("Spark:", spark.version)
print("Scala JVM:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

Spark: 4.1.1
Scala JVM: 2.13.17


## Paso 1. Leer el stream desde Kafka

Aquí todavía no parseamos el JSON. Solo leemos los mensajes crudos del topic.

In [3]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "ec-kafka:9092") \
    .option("subscribe", "orden-eventos") \
    .option("startingOffsets", "earliest") \
    .load()

In [4]:
df_kafka.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## Paso 2. Convertir `value` a texto (Saltar al paso 3)

Kafka entrega `key` y `value` en binario. Por eso primero convertimos `value` a `STRING`.

In [ ]:
#df_texto = df_kafka.selectExpr("CAST(value AS STRING) AS mensaje")
#df_texto.printSchema()

In [ ]:
# Vista rápida del mensaje en texto
# Si quieres depurar primero sin parsear JSON, ejecuta esta celda.
#query_texto = df_texto.writeStream \
#    .format("console") \
#    .outputMode("append") \
#    .option("truncate", False) \
#    .start()

Cuando termines de ver mensajes crudos, detén la consulta antes de seguir:

In [ ]:
#query_texto.stop()

## Paso 3. Definir el esquema común del evento

Este contrato permite que Spark interprete igual los mensajes vengan de Java o de Python.

### Evento esperado
```json
{
  "tipoEvento": "orden.creada",
  "ordenId": 1,
  "total": 150.0,
  "estado": "PENDIENTE",
  "origen": "java",
  "timestamp": 1710000000000
}
```

El campo `timestamp` esta en milisegundos Unix y permite calcular latencia.

In [5]:
from pyspark.sql.functions import col, current_timestamp, from_json, unix_millis
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

schema = StructType([
    StructField("tipoEvento", StringType(), True),
    StructField("ordenId", LongType(), True),
    StructField("total", DoubleType(), True),
    StructField("estado", StringType(), True),
    StructField("origen", StringType(), True),
    StructField("timestamp", LongType(), True)
])

## Paso 4. Parsear el JSON

In [6]:
df_value = df_kafka.selectExpr(
    "CAST(value AS STRING) AS value",
    "topic",
    "partition",
    "offset",
    "timestamp AS kafkaTimestamp"
)

df_parsed = df_value.select(
    "topic",
    "partition",
    "offset",
    "kafkaTimestamp",
    from_json(col("value"), schema).alias("data")
).select("topic", "partition", "offset", "kafkaTimestamp", "data.*")

In [7]:
df_parsed.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- ordenId: long (nullable = true)
 |-- total: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)



## Paso 5. Validar registros

Filtramos mensajes que sí tengan al menos el contrato básico:
- `tipoEvento`
- `ordenId`
- `total`
- `estado`

In [8]:
validity_condition = (
    col("tipoEvento").isNotNull() &
    col("ordenId").isNotNull() &
    col("total").isNotNull() &
    col("estado").isNotNull() &
    col("timestamp").isNotNull()
)

df_observable = df_parsed.withColumn("isValid", validity_condition)
df_validado = df_observable.filter(col("isValid"))

## Paso 6. Transformación de ejemplo

En este caso, solo dejamos órdenes con `total > 100`.

In [9]:
df_transformed = df_validado.filter(col("total") > 0) \
    .withColumn("processedAt", unix_millis(current_timestamp())) \
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))

## Paso 7. Salida a consola

Salida visual simple para confirmar que Spark recibe eventos. Mantiene el comportamiento original del notebook.

In [10]:
# Momento 1: Consola debug visual
query_console = df_transformed.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

# query_console.awaitTermination()

26/05/04 02:55:10 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-a937e088-43e7-45e0-8eb9-4d6a87b31eb7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/04 02:55:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total |estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |0     |2026-05-03 17:31:09.255|orden.creada|22     |223.0 |PENDIENTE|python     |1777829468956|true   |1777863311715|33842759 |
|orden-eventos|0        |1     |2026-05-03 17:31:11.282|orden.creada|811    |333.0 |PENDIENTE|python     |1777829471282|true   |1777863311715|33840433 |
|orden-eventos|0        |2     |2026-05-03 17:31:13.289|orden.creada|992    |343.0 |PENDIENTE|python     |1777829473289|true   |1777863311

## Detener la consulta
Si necesitas detener manualmente la ejecución, puedes usar:

In [11]:
#query_console.stop()

## Ejemplos de productores compatibles

### Java
```java
@NoArgsConstructor
@AllArgsConstructor
public class EventoOrden {
    private String tipoEvento;
    private Long ordenId;
    private Double total;
    private String estado;
}
```

### Python
```python
from kafka import KafkaProducer
import json, time, random

producer = KafkaProducer(
    bootstrap_servers='ec-kafka:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

while True:
    data = {
        "tipoEvento": "orden.creada",
        "ordenId": random.randint(1, 1000),
        "total": random.randint(50, 1500),
        "estado": "PENDIENTE",
        "origen": "python",
        "timestamp": int(time.time() * 1000)
    }

    producer.send("orden-eventos", value=data)
    print("Enviado:", data)
    time.sleep(2)
```

In [11]:
# Momento 2: persistencia Arrancar stream a Parquet o Salida a Parquet

# Si esta celda se vuelve a ejecutar, detenemos solo la query Parquet anterior.
try:
    query_parquet.stop()
    print("query_parquet anterior detenida.")
except NameError:
    print("No habia query_parquet previa.")
except Exception as e:
    print(f"query_parquet previa no estaba activa: {e}")

query_parquet = df_transformed.writeStream \
    .queryName("orden_eventos_parquet_observable") \
    .format("parquet") \
    .option("path", "/opt/artifacts/output/orden_eventos_parquet") \
    .option("checkpointLocation", "/opt/artifacts/checkpoint/orden-eventos-parquet-observable") \
    .outputMode("append") \
    .start()


No habia query_parquet previa.


26/05/04 02:55:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [12]:
#Esperar micro-batch
#import time
#time.sleep(10)



In [13]:
# Ver métricas del último micro-batch de la consola
progress = query_console.lastProgress
progress

{
    "id": "7190e2c4-3ba7-49f3-9128-82f2f7585710",
    "runId": "1d1f664a-5687-4f3b-a730-1629028abe56",
    "name": null,
    "timestamp": "2026-05-04T02:55:55.240Z",
    "batchId": 1,
    "batchDuration": 7,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 7,
        "triggerExecution": 7
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "KafkaV2[Subscribe[orden-eventos]]",
            "startOffset": {
                "orden-eventos": {
                    "0": 13
                }
            },
            "endOffset": {
                "orden-eventos": {
                    "0": 13
                }
            },
            "latestOffset": {
                "orden-eventos": {
                    "0": 13
                }
            },
            "numInputRows": 0,
            "inputRowsPerSecond": 0.0,
            "processedRowsPerSecond": 0.0,
            "

In [14]:
# Leer lo guardado
df_final = spark.read.parquet("/opt/artifacts/output/orden_eventos_parquet")

df_final.show(200, truncate=False)

+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total |estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+------+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |0     |2026-05-03 17:31:09.255|orden.creada|22     |223.0 |PENDIENTE|python     |1777829468956|true   |1777862798064|33329108 |
|orden-eventos|0        |1     |2026-05-03 17:31:11.282|orden.creada|811    |333.0 |PENDIENTE|python     |1777829471282|true   |1777862798064|33326782 |
|orden-eventos|0        |2     |2026-05-03 17:31:13.289|orden.creada|992    |343.0 |PENDIENTE|python     |1777829473289|true   |1777862798064|33324775 |
|orden-eventos|0        |3     |2026-05-03 17:31:15.294|orden.creada|718    |325.0

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total|estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |13    |2026-05-04 02:56:16.468|orden.creada|37     |700.0|PENDIENTE|ec-orden-ms|1777863376446|true   |1777863376626|180      |
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|topic        |partition|offset|kafkaTimestamp         |tipoEvento  |ordenId|total|estado   |origen     |timestamp    |isValid|processedAt  |latencyMs|
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+
|orden-eventos|0        |14    |2026-05-04 02:56:49.586|orden.creada|38     |700.0|PENDIENTE|ec-orden-ms|1777863409586|true   |1777863409834|248      |
+-------------+---------+------+-----------------------+------------+-------+-----+---------+-----------+-------------+-------+-------------+---------+



In [ ]:
df_final.printSchema()

In [ ]:
df_final.select("tipoEvento", "ordenId", "total", "estado").show()

In [ ]:
# Detener stream cuando ya no lo necesites, Detener ambas
# query_console.stop()
# query_parquet.stop()